# NCKH V7.4 — High-Resolution 8×8 Tiling Pilot (exploratory)

**Mục tiêu:** Kiểm tra xem tăng độ phân giải phân vùng từ 4×4 lên **8×8** có giúp CountGD++ tìm thêm các đối tượng nhỏ trong ảnh đông đúc hay không. Tận dụng kết quả **V7.3 đã hoàn tất**; **không chạy lại 1.301 ảnh**.

**Điều kiện vào nhánh V7.4 (KHÔNG dùng ground truth):** CountGD++ toàn ảnh đếm 0; dự đoán V7.3 ≥ 150; trước đó target đã được V7.2 hoặc V7.3 Qwen xác nhận; có ít nhất 150 box tiling 4×4. Trên cache hiện tại dự kiến có **6 ảnh**. Phần GT chỉ được truy cập sau khi chốt dự đoán.

**Quy tắc an toàn:** Chỉ chấp nhận số đếm 8×8 khi Qwen mới xác nhận ≥4/6 box ngữ cảnh, đủ 6 nhãn hợp lệ, số đếm mới không thấp hơn V7.3, tăng không quá 3×, và đã có bằng chứng xác nhận target trước đó. Bất kỳ lỗi/không chắc chắn nào => giữ nguyên V7.3.

**Quan trọng:** Đây là thử nghiệm *post-hoc exploratory* trên tập TEST đã được dùng nhiều lần để phát triển; không đủ để tuyên bố tính tổng quát hóa. Các chỉ số negative có thể không đổi vì bộ router này không kích hoạt trên negative có baseline 0 trong dữ liệu hiện tại.

Notebook yêu cầu: Kaggle GPU + Internet; Add Input FSC147 images (`hihihi5`) và `V7_4_Inputs_Ready_Kaggle.zip`. Chạy Run All một lần. Có ZIP checkpoint sau từng ảnh.

**Kaggle fix (Nov 2026):** Added a preflight cell `02B` for the `transformers` stale `tensorflow_text` backend issue. The dependency cell pins `transformers==4.57.1`; preflight only reinstalls it if import fails. No changes to V7.4 inference/routing, data, or metrics.


In [ ]:
# 01 — CONFIG; freeze protocol before assessing labels
import os, sys, time, json, re, math, io, gc, zipfile, shutil, hashlib, subprocess
from pathlib import Path
import numpy as np, pandas as pd
from PIL import Image, ImageDraw, ImageOps
SEED=42; np.random.seed(SEED)
ROOT=Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('/mnt/data')
OUT=ROOT/'v74_high_res'
OUT.mkdir(parents=True,exist_ok=True)
INPUT_ROOTS=[p for p in [Path('/kaggle/input'),Path('/mnt/data')] if p.exists()]
DATA_DIR=OUT/'inputs'; DATA_DIR.mkdir(exist_ok=True)
BOX_DIR=OUT/'grid8_boxes'; BOX_DIR.mkdir(exist_ok=True)
TILE_CKPT=OUT/'v74_grid8_checkpoint.csv'
GATE_CKPT=OUT/'v74_qwen_checkpoint.csv'
COUNTGD_REPO=ROOT/'CountGDPlusPlus'
COUNTGD_GITHUB='https://github.com/niki-amini-naieni/CountGDPlusPlus.git'
COUNTGD_GDRIVE_ID='1j6N22TtKu2NVcKpgfrf-sJHGeLDqs9hs'
QWEN_MODEL_ID='Qwen/Qwen3-VL-4B-Instruct'
COUNTGD_THRESH=.23
TILE_OVERLAP_FRAC=.20
NMS_IOU=.50
FINE_GRID=8
N_SAMPLE=6
MIN_LOCAL_YES=4
CONTEXT_MULTIPLIER=6.0
PANEL_SIZE=336
MAX_NEW_RATIO=3.0
MAX_IMAGES=None # None = tất cả các ảnh eligible; chỉ dùng smoke khi cần
print('Output',OUT)

In [ ]:
# 02 — INPUT ZIP/CSV tolerant to Kaggle auto-unzip
needed=('v73_1301_predictions.csv','v73_verification_audit.csv','v72_reparsed_visual_gate_27.csv')
for name in needed:
    if (DATA_DIR/name).exists(): continue
    for parent in INPUT_ROOTS:
        hits=sorted(parent.rglob(name),key=str)
        if hits:
            (DATA_DIR/name).write_bytes(hits[0].read_bytes())
            break
if any(not (DATA_DIR/n).exists() for n in needed):
    for parent in INPUT_ROOTS:
        for zpath in sorted(parent.rglob('*.zip'),key=str):
            try:
                with zipfile.ZipFile(zpath) as z:
                    for name in needed:
                        if not (DATA_DIR/name).exists() and name in z.namelist():
                            (DATA_DIR/name).write_bytes(z.read(name))
            except (zipfile.BadZipFile, PermissionError):pass
missing=[n for n in needed if not (DATA_DIR/n).exists()]
if missing:raise FileNotFoundError('Thiếu '+str(missing)+'. Add Input V7_4_Inputs_Ready_Kaggle.zip')
base=pd.read_csv(DATA_DIR/'v73_1301_predictions.csv')
v72_gates=pd.read_csv(DATA_DIR/'v72_reparsed_visual_gate_27.csv')
v73_gates=pd.read_csv(DATA_DIR/'v73_verification_audit.csv')
assert len(base)==1301 and base.group_id.is_unique
assert not base[['baseline_count','v73_count']].isna().any().any()
for f in [v72_gates,v73_gates]:assert f.group_id.is_unique
accepted_v72=set(v72_gates.loc[v72_gates.gate_fixed.eq('CONFIRMED'),'group_id'])
accepted_v73=set(v73_gates.loc[v73_gates.accepted.astype(str).str.lower().eq('true'),'group_id'])
verified=accepted_v72 | accepted_v73
old_count=v72_gates.set_index('group_id')['tile_count'].to_dict()
probe=base.loc[base.baseline_count.eq(0)&base.v73_count.ge(150)&base.group_id.isin(verified),['group_id','image_name','target','baseline_count','v73_count']].copy()
probe['old_tiled_count']=probe.group_id.map(old_count)
probe=probe.loc[probe.old_tiled_count.ge(150)].sort_values('group_id').reset_index(drop=True)
if MAX_IMAGES is not None:probe=probe.head(MAX_IMAGES).copy()
assert len(probe)>0,'Không có ảnh đủ điều kiện theo quy tắc V7.4'
# Explicit no GT usage for eligibility decisions
assert 'gt_count' not in probe.columns
root_imgs=Path('/kaggle/input') if Path('/kaggle/input').exists() else Path('/mnt/data')
images={}
for p in root_imgs.rglob('*.jpg'):
    images.setdefault(p.name,str(p))
missing_imgs=sorted(set(probe.image_name)-set(images))
if missing_imgs:raise FileNotFoundError('Thiếu ảnh FSC-147 '+str(missing_imgs[:8]))
probe['image_path']=probe.image_name.map(images)
print('Số ảnh chạy 8x8:',len(probe))
print(probe[['image_name','target','baseline_count','v73_count','old_tiled_count']].to_string(index=False))

In [ ]:
# ============================================================
# CELL 2 — DEPENDENCIES + COUNTGD++ REPO
# Giữ Torch/Torchvision hiện tại.
# ============================================================

import sys
import subprocess
import importlib.metadata as im
from pathlib import Path

def pkg_version(name):
    try:
        return im.version(name)
    except Exception:
        return None

print("Existing torch:", pkg_version("torch"))
print("Existing torchvision:", pkg_version("torchvision"))
print("Existing transformers:", pkg_version("transformers"))

if not COUNTGD_REPO.exists():
    subprocess.run(
        ["git", "clone", "-q", COUNTGD_GITHUB, str(COUNTGD_REPO)],
        check=True,
    )

req_file = COUNTGD_REPO / "requirements.txt"
if req_file.exists():
    blocked = (
        "torch", "torchvision", "torchaudio", "numpy", "scipy", "opencv",
        "gradio", "spaces", "deeplake"
    )
    keep = []
    for line in req_file.read_text(encoding="utf-8").splitlines():
        s = line.strip()
        if not s or s.startswith("#"):
            continue
        if any(s.lower().startswith(x) for x in blocked):
            continue
        keep.append(s)

    filtered = Path("/kaggle/working/countgd_v4_requirements_filtered.txt")
    filtered.write_text("\n".join(keep) + "\n", encoding="utf-8")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(filtered)],
        check=True,
    )

subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q", "--upgrade",
        "transformers==4.57.1",
        "accelerate>=1.2.0",
        "sentencepiece",
        "gdown",
        "qwen-vl-utils",
    ],
    check=True,
)

print("Dependency setup finished.")
print("transformers:", pkg_version("transformers"))


In [ ]:
# 02B — REPAIR HUGGING FACE TRANSFORMERS INSTALLATION BEFORE COUNTGD++
# Fix for: ValueError: Offending backend: tensorflow_text
# Do NOT install tensorflow-text. Keep Torch/Torchvision unchanged.
import os, sys, site, sysconfig, subprocess, importlib, shutil
from pathlib import Path
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"

PINNED_TRANSFORMERS = "4.57.1"
probe_code = (
    'import os; os.environ["USE_TF"]="0"; os.environ["USE_FLAX"]="0"; '
    'import transformers; '
    'from transformers import BertModel, BertTokenizer, Qwen3VLForConditionalGeneration; '
    'print("OK transformers", transformers.__version__)'
)

def test_import():
    return subprocess.run(
        [sys.executable, "-c", probe_code],
        text=True, capture_output=True,
        env={**os.environ, "USE_TF": "0", "USE_FLAX": "0"},
    )

check = test_import()
if check.returncode != 0:
    print("Transformers import failed; cleaning old package files ...")
    print("\n".join((check.stderr or "").splitlines()[-7:]))
    subprocess.run(
        [sys.executable, "-m", "pip", "uninstall", "-y", "transformers"],
        check=True,
    )
    # Pip uninstall may leave obsolete files (e.g. tokenization_bert_tf.py).
    # ONLY remove the transformers package folder in Python site-packages.
    site_roots = set(site.getsitepackages())
    site_roots.add(sysconfig.get_paths()["purelib"])
    site_roots.add(site.getusersitepackages())
    for root in sorted(site_roots):
        folder = Path(root) / "transformers"
        if folder.is_dir() and folder.parent.name in ("site-packages", "dist-packages"):
            print("Removing stale files:", folder)
            shutil.rmtree(folder)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir",
         "--no-deps", f"transformers=={PINNED_TRANSFORMERS}"],
        check=True,
    )
    importlib.invalidate_caches()
    # Clear partial module objects from a prior failed lazy import.
    for mod in list(sys.modules):
        if mod == "transformers" or mod.startswith("transformers."):
            del sys.modules[mod]
    check = test_import()

if check.returncode != 0:
    print("\n".join((check.stderr or "").splitlines()[-28:]))
    raise RuntimeError("Transformers vẫn không import được; gửi log của cell này.")
print("PASS:", check.stdout.strip())
print("Torch/Torchvision were not reinstalled.")


In [ ]:
# ============================================================
# CELL 3 — VERIFY TORCH/CUDA + PATCH/BUILD GROUNDINGDINO OPS
# ============================================================

import os
import re
import sys
import gc
import importlib
import subprocess
from pathlib import Path

import torch
import torchvision

print("Torch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("Torch CUDA:", torch.version.cuda)
print("GPU count:", torch.cuda.device_count())

if not torch.cuda.is_available():
    raise RuntimeError("V3 cần GPU Kaggle.")

for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

from torchvision.ops import nms
_test_boxes = torch.tensor([[0.,0.,10.,10.],[1.,1.,9.,9.]], device="cuda")
_test_scores = torch.tensor([0.9,0.8], device="cuda")
_ = nms(_test_boxes, _test_scores, 0.5)
print("torchvision CUDA NMS: PASS")

def detectron2_ok():
    try:
        import detectron2
        from detectron2 import _C
        return True
    except Exception as e:
        print("Detectron2 check:", repr(e))
        return False

if not detectron2_ok():
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q", "--no-build-isolation",
            "git+https://github.com/facebookresearch/detectron2.git",
        ],
        check=True,
    )
    importlib.invalidate_caches()
    if not detectron2_ok():
        raise RuntimeError("Detectron2 vẫn không load được sau khi cài.")

OPS_DIR = COUNTGD_REPO / "models" / "GroundingDINO" / "ops"
SRC_DIR = OPS_DIR / "src"
assert OPS_DIR.exists(), OPS_DIR

source_suffixes = {".h", ".hpp", ".cpp", ".cu", ".cuh"}
patched = []
for path in SRC_DIR.rglob("*"):
    if not path.is_file() or path.suffix not in source_suffixes:
        continue
    original = path.read_text(encoding="utf-8", errors="ignore")
    updated = original
    updated = re.sub(
        r"([A-Za-z_][A-Za-z0-9_]*)\.type\(\)\.is_cuda\(\)",
        r"\1.is_cuda()",
        updated,
    )
    updated = re.sub(
        r"([A-Za-z_][A-Za-z0-9_]*)\.type\(\)\.scalar_type\(\)",
        r"\1.scalar_type()",
        updated,
    )
    updated = updated.replace("value.type()", "value.scalar_type()")
    if updated != original:
        path.write_text(updated, encoding="utf-8")
        patched.append(str(path.relative_to(OPS_DIR)))

print("Patched files:", patched if patched else "already compatible")

from torch.utils.cpp_extension import CUDA_HOME
if CUDA_HOME:
    os.environ["CUDA_HOME"] = str(CUDA_HOME)

major, minor = torch.cuda.get_device_capability(0)
os.environ["TORCH_CUDA_ARCH_LIST"] = f"{major}.{minor}"
os.environ["MAX_JOBS"] = "1"

if Path("/usr/bin/gcc-11").exists():
    os.environ["CC"] = "/usr/bin/gcc-11"
if Path("/usr/bin/g++-11").exists():
    os.environ["CXX"] = "/usr/bin/g++-11"

def try_import_msda():
    try:
        if str(OPS_DIR) not in sys.path:
            sys.path.insert(0, str(OPS_DIR))
        importlib.invalidate_caches()
        return importlib.import_module("MultiScaleDeformableAttention")
    except Exception:
        return None

MSDA = try_import_msda()
if MSDA is None:
    print("Building GroundingDINO CUDA ops...")
    subprocess.run(
        [sys.executable, "setup.py", "build_ext", "--inplace"],
        cwd=str(OPS_DIR),
        check=True,
    )
    importlib.invalidate_caches()
    sys.modules.pop("MultiScaleDeformableAttention", None)
    MSDA = try_import_msda()

if MSDA is None:
    raise RuntimeError("Không import được MultiScaleDeformableAttention sau build.")

print("GroundingDINO CUDA ops: PASS", MSDA.__file__)


In [ ]:
# ============================================================
# CELL 4 — DOWNLOAD COUNTGD++ WEIGHTS
# ============================================================

import sys
import subprocess
from pathlib import Path

CHECKPOINT_DIR = COUNTGD_REPO / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

BERT_DIR = CHECKPOINT_DIR / "bert-base-uncased"
if not (BERT_DIR / "config.json").exists():
    print("Downloading bert-base-uncased...")
    subprocess.run(
        [sys.executable, "download_bert.py"],
        cwd=str(COUNTGD_REPO),
        check=True,
    )

COUNTGD_CKPT = CHECKPOINT_DIR / "countgd_plusplus.pth"
if (not COUNTGD_CKPT.exists()) or COUNTGD_CKPT.stat().st_size < 500 * 1024**2:
    print("Downloading CountGD++ checkpoint...")
    subprocess.run(
        [sys.executable, "-m", "gdown", COUNTGD_GDRIVE_ID, "-O", str(COUNTGD_CKPT)],
        check=True,
    )

assert (BERT_DIR / "config.json").exists()
assert COUNTGD_CKPT.exists()
print("BERT:", BERT_DIR)
print("CountGD++ checkpoint GB:", round(COUNTGD_CKPT.stat().st_size / 1024**3, 2))


In [ ]:
# 05B — RESET TRANSFORMERS smart_apply WHEN NOTEBOOK SWITCHED VERSIONS
# CountGD++ uses BertModel.from_pretrained; transformers 4.57.1 expects
# smart_apply(self, fn). A newer Kaggle version may have previously defined
# smart_apply(self, fn, is_custom_code) on torch.nn.Module.
# This temporary method survives pip reinstall in an active Python kernel.
import torch, inspect, transformers
print('Current transformers:', transformers.__version__)
if transformers.__version__ != '4.57.1':
    raise RuntimeError(
        'Notebook V7.4 expects transformers==4.57.1. '
        'Please rerun the installation/repair cell before loading CountGD++.'
    )
if 'smart_apply' in vars(torch.nn.Module):
    print('Removing previous smart_apply signature:',
          inspect.signature(torch.nn.Module.smart_apply))
    delattr(torch.nn.Module, 'smart_apply')
else:
    print('No stale smart_apply found.')
print('PASS: BertModel will register matching smart_apply on next load.')


In [ ]:
# 06 — LOAD COUNTGD++ (giữ nguyên API cũ)
import torch, importlib
# Reset stale Transformers smart_apply, e.g. new-version (self, fn, is_custom_code)
# persists in torch.nn.Module after pip installs transformers 4.57.1.
if 'smart_apply' in vars(torch.nn.Module):
    delattr(torch.nn.Module, 'smart_apply')
import gc
p=str(COUNTGD_REPO)
if p not in sys.path:sys.path.insert(0,p)
os.chdir(str(COUNTGD_REPO))
for name in list(sys.modules):
    if name=='models' or name.startswith('models.'):
        del sys.modules[name]
importlib.invalidate_caches();gc.collect();torch.cuda.empty_cache()
process_file=COUNTGD_REPO/'process_folder.py'
source=process_file.read_text(encoding='utf-8')
marker='logging.basicConfig(level=logging.INFO'
if marker not in source:raise RuntimeError('API CountGD++ thay đổi')
countgd_namespace={'__name__':'v72_countgd_runtime','__file__':str(process_file)}
exec(compile(source.split(marker,1)[0],str(process_file),'exec'),countgd_namespace)
for k in ['build_arg_parser','build_model_and_transforms','count']:
    if k not in countgd_namespace:raise RuntimeError(f'Missing CountGD++ API: {k}')
args=countgd_namespace['build_arg_parser']().parse_args([])
args.device='cuda:0';args.pretrain_model_path=str(COUNTGD_CKPT)
_load=torch.load
def _load_trusted(*a,**kw):
    kw.setdefault('weights_only',False)
    return _load(*a,**kw)
torch.load=_load_trusted
try:
    countgd_model,countgd_transform=countgd_namespace['build_model_and_transforms'](args)
finally:
    torch.load=_load
countgd_model=countgd_model.to('cuda:0').eval()
countgd_namespace.update(model=countgd_model,transform=countgd_transform,device=torch.device('cuda:0'))
print('CountGD++ loaded')


In [ ]:
# 07 — HELPERS: TILE, TỌA ĐỘ TOÀN ẢNH, NMS, CHECKPOINT

def atomic_csv(df,path):
    temp=Path(str(path)+'.tmp')
    df.to_csv(temp,index=False)
    temp.replace(path)

def iou(a,b):
    ax1,ay1,ax2,ay2=a; bx1,by1,bx2,by2=b
    inter=max(0,min(ax2,bx2)-max(ax1,bx1))*max(0,min(ay2,by2)-max(ay1,by1))
    area_a=max(0,ax2-ax1)*max(0,ay2-ay1)
    area_b=max(0,bx2-bx1)*max(0,by2-by1)
    return inter/max(1e-9,area_a+area_b-inter)

def nms_rows(rows,thr=NMS_IOU):
    chosen=[]
    for row in sorted(rows,key=lambda r:-float(r['score'])):
        if not any(iou(row['box'],prev['box'])>thr for prev in chosen):
            chosen.append(row)
    return chosen

def count_tiled(img,target,grid):
    W,H=img.size
    detections=[]
    for gy in range(grid):
        for gx in range(grid):
            core_x0=int(round(gx*W/grid));core_x1=int(round((gx+1)*W/grid))
            core_y0=int(round(gy*H/grid));core_y1=int(round((gy+1)*H/grid))
            dx=(core_x1-core_x0)*TILE_OVERLAP_FRAC
            dy=(core_y1-core_y0)*TILE_OVERLAP_FRAC
            x0=max(0,int(math.floor(core_x0-dx)))
            x1=min(W,int(math.ceil(core_x1+dx)))
            y0=max(0,int(math.floor(core_y0-dy)))
            y1=min(H,int(math.ceil(core_y1+dy)))
            crop=img.crop((x0,y0,x1,y1))
            if min(crop.size)<8:continue
            boxes,scores,_=countgd_namespace['count'](crop,str(target).replace('.','').strip(),{'image':crop,'points':[]},[],[],float(COUNTGD_THRESH),False)
            boxes=boxes.tolist() if hasattr(boxes,'tolist') else list(boxes)
            scores=scores.tolist() if hasattr(scores,'tolist') else list(scores)
            for box,score in zip(boxes,scores):
                score=float(score)
                if score<=COUNTGD_THRESH:continue
                xx0,yy0,xx1,yy1=[float(v) for v in box[:4]]
                # CountGD++ gốc trả bbox xyxy theo pixel của PIL image.
                xx0=min(max(xx0+x0,0),W);xx1=min(max(xx1+x0,0),W)
                yy0=min(max(yy0+y0,0),H);yy1=min(max(yy1+y0,0),H)
                bx0,bx1=sorted([xx0,xx1]);by0,by1=sorted([yy0,yy1])
                cx=(bx0+bx1)/2;cy=(by0+by1)/2
                # Ownership by center, tránh đếm hai lần ở các ô overlap.
                inside_x = core_x0<=cx<(core_x1 if gx<grid-1 else core_x1+1)
                inside_y = core_y0<=cy<(core_y1 if gy<grid-1 else core_y1+1)
                if inside_x and inside_y and bx1>bx0 and by1>by0:
                    detections.append({'box':[bx0,by0,bx1,by1],'score':score,'grid':grid,'cell':[gx,gy]})
    return nms_rows(detections)

# small self-test of NMS and IoU
assert iou([0,0,10,10],[0,0,10,10])>0.999
assert len(nms_rows([{'box':[0,0,10,10],'score':.9},{'box':[1,1,10,10],'score':.8}]))==1
print('Helpers ready; NMS self-test PASS')

In [ ]:
# 08 — 8×8 inference; checkpoint after each image
# Keep Checkpoint names distinct from V7.2; do not overwrite any original result

def restore_file(filename, dest):
    if dest.exists():return
    for root in INPUT_ROOTS:
        hits=sorted(root.rglob(filename),key=str)
        if hits:shutil.copyfile(hits[0],dest);return

def snapshot():
    dst=ROOT/'V7_4_HighRes_Partial_Checkpoint.zip'
    with zipfile.ZipFile(dst,'w',zipfile.ZIP_DEFLATED) as zz:
        for p in (TILE_CKPT,GATE_CKPT):
            if p.exists():zz.write(p,p.name)
        for p in BOX_DIR.glob('*.json'):zz.write(p,'grid8_boxes/'+p.name)
    return dst

restore_file(TILE_CKPT.name,TILE_CKPT)
if not any(BOX_DIR.iterdir()):
    for root in INPUT_ROOTS:
        for zpath in sorted(root.rglob('V7_4_HighRes_Partial_Checkpoint.zip'),key=str):
            with zipfile.ZipFile(zpath) as zz:
                for nm in zz.namelist():
                    if nm.startswith('grid8_boxes/') and nm.endswith('.json'):
                        (BOX_DIR/Path(nm).name).write_bytes(zz.read(nm))
rows=pd.read_csv(TILE_CKPT).to_dict('records') if TILE_CKPT.exists() else []
done={str(r['group_id']) for r in rows if r['status']=='ok' and (BOX_DIR/(str(r['group_id'])+'.json')).exists()}
for i,r in enumerate(probe.itertuples(index=False),1):
    gid=r.group_id
    if gid in done:
        print(f'[SKIP] {i}/{len(probe)} {gid}');continue
    st=time.time()
    try:
        img=Image.open(r.image_path).convert('RGB')
        boxes=count_tiled(img,r.target,FINE_GRID)
        (BOX_DIR/(gid+'.json')).write_text(json.dumps(boxes),encoding='utf-8')
        row={'group_id':gid,'image_name':r.image_name,'target':r.target,
             'old_count':int(r.v73_count),'grid8_count':len(boxes),'nms_iou':NMS_IOU,
             'elapsed_sec':round(time.time()-st,2),'status':'ok','error':''}
    except Exception as e:
        row={'group_id':gid,'image_name':r.image_name,'target':r.target,
             'old_count':int(r.v73_count),'grid8_count':-1,'nms_iou':NMS_IOU,
             'elapsed_sec':round(time.time()-st,2),'status':'error','error':repr(e)}
    rows=[x for x in rows if x['group_id']!=gid]+[row]
    atomic_csv(pd.DataFrame(rows),TILE_CKPT)
    print(f'[GRID8] {i}/{len(probe)} {gid}: {row["grid8_count"]}, sec={row["elapsed_sec"]}')
    print('Checkpoint:',snapshot())
    if row['status']!='ok':raise RuntimeError('Grid8 lỗi: '+row['error'])
assert set(probe.group_id).issubset(set(pd.read_csv(TILE_CKPT).loc[lambda d:d.status.eq('ok'),'group_id']))
print('Done GRID8')

In [ ]:
# 09 — free CountGD++ before loading Qwen3-VL
import torch,gc
for name in ('countgd_model','countgd_transform'):
    if name in globals():del globals()[name]
if 'countgd_namespace' in globals():
    countgd_namespace.pop('model',None)
    countgd_namespace.pop('transform',None)
gc.collect();torch.cuda.empty_cache()
print('CUDA cache cleared')

In [ ]:
# CELL 3 — QWEN3-VL: LOAD PROCESSOR FROM LOCAL CONFIG FILES
# Avoid 404 on HF Hub optional additional_chat_templates directory.
# Keep Transformers 4.57.1: do not upgrade it in this cell.
import torch, gc, sys, subprocess
from huggingface_hub import snapshot_download
from transformers import AutoProcessor, Qwen3VLForConditionalGeneration

assert torch.cuda.is_available(), "Hãy bật GPU trong Kaggle Session options."
try:
    from qwen_vl_utils import process_vision_info
except ModuleNotFoundError:
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', 'qwen-vl-utils'],
        check=True,
    )
    from qwen_vl_utils import process_vision_info

# Only small processor/tokenizer files, NOT 9GB model weights.
processor_dir = snapshot_download(
    repo_id=QWEN_MODEL_ID,
    allow_patterns=[
        'config.json',
        'processor_config.json',
        'preprocessor_config.json',
        'video_preprocessor_config.json',
        'tokenizer.json',
        'tokenizer_config.json',
        'vocab.json',
        'merges.txt',
        'chat_template.json',
        'chat_template.jinja',
        'special_tokens_map.json',
        'added_tokens.json',
    ],
)

# A local path prevents unnecessary HF Hub queries for a missing
# optional folder named additional_chat_templates.
processor = AutoProcessor.from_pretrained(
    processor_dir,
    local_files_only=True,
)
print('Qwen processor loaded from:', processor_dir)

# Download/load model weights normally; model ID is valid.
model = Qwen3VLForConditionalGeneration.from_pretrained(
    QWEN_MODEL_ID,
    dtype=torch.float16,
    device_map='auto',
    low_cpu_mem_usage=True,
).eval()
print('Loaded Qwen:', QWEN_MODEL_ID, 'GPU:', torch.cuda.get_device_name(0))


In [ ]:
# CELL 4 — CONTEXT PANELS & OUTPUT PARSER (unit tests)
from PIL import Image, ImageDraw, ImageOps

LINE_RX=re.compile(r'^(?:ID\s*)?(\d+)\s*[|:]\s*(YES|NO|UNCERTAIN)\s*$',re.I)
FULL_RX=re.compile(r'\b(?:YES|NO|UNCERTAIN)\b',re.I)

def parse_full(text):
    tags=FULL_RX.findall(str(text).upper())
    return tags[0] if len(tags)==1 else 'UNCERTAIN'

def parse_panel(text,n):
    out={}
    for line in str(text).strip().splitlines():
        match=LINE_RX.match(line.strip())
        if match:
            idx=int(match.group(1))
            if 0<=idx<n and idx not in out:out[idx]=match.group(2).upper()
    return out

def choose_diverse(boxes,n=N_SAMPLE):
    boxes=sorted(boxes,key=lambda x:-float(x.get('score',0)))
    chosen=[];used_cells=set()
    for x in boxes:
        cell=tuple(x.get('cell',[0,0]))
        if cell not in used_cells:
            chosen.append(x);used_cells.add(cell)
        if len(chosen)>=n:return chosen
    used_ids={id(x) for x in chosen}
    for x in boxes:
        if id(x) not in used_ids:
            chosen.append(x)
            if len(chosen)>=n:break
    return chosen

def local_context_box(im_size, bbox, mult=CONTEXT_MULTIPLIER):
    W,H=im_size
    x1,y1,x2,y2=map(float,bbox)
    cx=(x1+x2)/2;cy=(y1+y2)/2
    bw=max(1,x2-x1);bh=max(1,y2-y1)
    crop_w=max(mult*bw,40);crop_h=max(mult*bh,40)
    # Ít nhất một bề rộng nhỏ vừa đủ nhìn rõ đối tượng nhưng không phóng quá 50% ảnh.
    crop_w=min(W,max(crop_w,2*bw));crop_h=min(H,max(crop_h,2*bh))
    left=max(0,min(W-crop_w,cx-crop_w/2));top=max(0,min(H-crop_h,cy-crop_h/2))
    return tuple(map(int,[round(left),round(top),round(left+crop_w),round(top+crop_h)]))

def sheet_context(image, chosen):
    n=len(chosen); cols=3;rows=(n+cols-1)//cols; head=32
    sheet=Image.new('RGB',(cols*PANEL_SIZE,rows*(PANEL_SIZE+head)),'white')
    draw=ImageDraw.Draw(sheet)
    for i,one in enumerate(chosen):
        box=one['box'];rect=local_context_box(image.size,box)
        crop=image.crop(rect).convert('RGB')
        panel=ImageOps.contain(crop,(PANEL_SIZE-12,PANEL_SIZE-12))
        x0=(i%cols)*PANEL_SIZE;y0=(i//cols)*(PANEL_SIZE+head)
        ox=x0+(PANEL_SIZE-panel.width)//2;oy=y0+(PANEL_SIZE-panel.height)//2
        sheet.paste(panel,(ox,oy))
        # Convert global box to panel space (contain preserves aspect ratio)
        scale=panel.width/max(1,crop.width)
        bx=[ox+(float(box[j])-rect[0])*scale if j%2==0 else oy+(float(box[j])-rect[1])*scale for j in range(4)]
        draw.rectangle(bx,outline='red',width=4)
        draw.rectangle((x0,y0+PANEL_SIZE,x0+PANEL_SIZE-1,y0+PANEL_SIZE+head-1),fill='white',outline='black',width=1)
        draw.text((x0+8,y0+PANEL_SIZE+5),f'ID {i} (red rectangle)',fill='black')
    return sheet

assert parse_full('YES')=='YES'
assert parse_full('NO')=='NO'
assert parse_full('YES, NO')=='UNCERTAIN'
assert parse_panel('0|YES\n1|NO\n2|UNCERTAIN',3)=={0:'YES',1:'NO',2:'UNCERTAIN'}
assert local_context_box((500,400),[10,10,20,20])[2]>10
print('Context + parser self-tests PASS')

In [ ]:
# CELL 5 — QWEN CONTEXTUAL VERIFICATION
@torch.inference_mode()
def qwen_context_check(image,target,chosen):
    sheet=sheet_context(image,chosen)
    prompt=(
        f'Target category: "{target}".\n'
        'Image 1 is the full original photograph. Image 2 has 6 numbered zoomed-in panels, each with a RED rectangle.\n'
        'For each panel, classify ONLY the object INSIDE THE RED rectangle. The rest of the panel provides visual context.\n'
        'Is the boxed object truly an instance of the target category? Similar-looking or related categories are NOT target.\n'
        'Do NOT label the panel positive merely because the target exists elsewhere in the full image.\n'
        'If the boxed object is unclear or cut off, output UNCERTAIN.\n'
        'Reply with EXACTLY six lines, one for each panel ID from 0 through 5. '
        'On each line write the ID, a vertical bar, and YOUR verdict YES/NO/UNCERTAIN. No explanations.'
    )
    messages=[{'role':'user','content':[{'type':'image','image':image},
                                        {'type':'image','image':sheet},
                                        {'type':'text','text':prompt}]}]
    text=processor.apply_chat_template(messages,tokenize=False,add_generation_prompt=True)
    imgs,vids,kwargs=process_vision_info(messages,image_patch_size=16,
                                        return_video_kwargs=True,return_video_metadata=True)
    metas=None
    if vids is not None:
        vids,metas=zip(*vids);vids=list(vids);metas=list(metas)
    inputs=processor(text=[text],images=imgs,videos=vids,video_metadata=metas,
                     padding=True,return_tensors='pt',do_resize=False,**kwargs)
    inputs=inputs.to(next(model.parameters()).device)
    gen=model.generate(**inputs,max_new_tokens=128,do_sample=False)
    raw=processor.batch_decode([out[len(inp):] for inp,out in zip(inputs.input_ids,gen)],
                               skip_special_tokens=True,clean_up_tokenization_spaces=False)[0]
    return raw,sheet

In [ ]:
# 13 — independent contextual Qwen verification of NEW 8x8 candidate boxes
restore_file(GATE_CKPT.name,GATE_CKPT)
prior=pd.read_csv(GATE_CKPT).to_dict('records') if GATE_CKPT.exists() else []
done={r['group_id'] for r in prior if r['status']=='ok'}
for i,r in enumerate(probe.itertuples(index=False),1):
    if r.group_id in done:
        print('[SKIP QWEN]',r.group_id);continue
    boxes=json.loads((BOX_DIR/(r.group_id+'.json')).read_text())
    st=time.time()
    try:
        img=Image.open(r.image_path).convert('RGB')
        chosen=choose_diverse(boxes,N_SAMPLE) if len(boxes)>=N_SAMPLE else []
        if len(chosen)==N_SAMPLE:
            raw,sheet=qwen_context_check(img,r.target,chosen)
            parsed=parse_panel(raw,N_SAMPLE)
            complete=len(parsed)==N_SAMPLE
            yes=sum(label=='YES' for label in parsed.values())
            if complete:assert sorted(parsed)==list(range(N_SAMPLE))
            sheet.save(OUT/(r.group_id+'_grid8_context.jpg'),quality=88)
        else:raw='';complete=False;yes=0
        nc=len(boxes)
        ratio=(nc/max(1,int(r.v73_count)))
        # Pre-existing independent full-image gate: verified V7.2 or V7.3.
        # Fresh Qwen confirmation of new boxes, no use of GT.
        accept=(complete and yes>=MIN_LOCAL_YES and
                nc>=int(r.v73_count) and nc<=MAX_NEW_RATIO*int(r.v73_count))
        new_count=nc if accept else int(r.v73_count)
        reason=('ACCEPT' if accept else 'REJECT_INSUFFICIENT_EVIDENCE_OR_OUT_OF_RANGE')
        record={'group_id':r.group_id,'image_name':r.image_name,'target':r.target,
                'old_count':int(r.v73_count),'grid8_count':nc,'ratio':round(ratio,4),
                'local_yes':yes,'complete':complete,'raw_context':raw,
                'accepted':accept,'v74_count':new_count,'reason':reason,
                'status':'ok','elapsed_sec':round(time.time()-st,2),'error':''}
    except Exception as e:
        record={'group_id':r.group_id,'image_name':r.image_name,'target':r.target,
                'old_count':int(r.v73_count),'grid8_count':len(boxes),'ratio':None,
                'local_yes':0,'complete':False,'raw_context':'',
                'accepted':False,'v74_count':int(r.v73_count),'reason':'ERROR_KEEP_V73',
                'status':'error','elapsed_sec':round(time.time()-st,2),'error':repr(e)}
    prior=[x for x in prior if x['group_id']!=r.group_id]+[record]
    atomic_csv(pd.DataFrame(prior),GATE_CKPT)
    print(f'[QWEN] {i}/{len(probe)} {r.group_id}: yes={record["local_yes"]} count={record["grid8_count"]} -> {record["v74_count"]}, gate={record["reason"]}')
    print('Checkpoint:',snapshot())
    if record['status']!='ok':raise RuntimeError('Qwen lỗi, giữ checkpoint để retry: '+record['error'])
assert set(probe.group_id).issubset(set(pd.read_csv(GATE_CKPT).loc[lambda d:d.status.eq('ok'),'group_id']))
print('Qwen stage complete')

In [ ]:
# 14 — LOCK inference first, THEN evaluate using GT
scores=pd.read_csv(GATE_CKPT)
assert scores.group_id.is_unique and set(scores.group_id)==set(probe.group_id)
pred=base.copy()
lookup=scores.set_index('group_id')['v74_count'].to_dict()
pred['v74_count']=pred.apply(lambda x:int(lookup.get(x.group_id,x.v73_count)),axis=1)
pred['changed_v74']=pred.v74_count.ne(pred.v73_count)
# Only now reference test GT to assess the locked predictions
metrics=[]
for subset,sub in [('POSITIVE',pred[pred.pair_type.eq('POSITIVE')]),
                   ('HARD_NEGATIVE',pred[pred.pair_type.eq('HARD_NEGATIVE')]),
                   ('ALL',pred)]:
    for name,col in [('V7.3','v73_count'),('V7.4_pilot','v74_count')]:
        err=(sub[col]-sub.gt_count).abs()
        metrics.append({'subset':subset,'method':name,'n':len(sub),'MAE':err.mean(),
                        'RMSE':np.sqrt(((sub[col]-sub.gt_count)**2).mean()),
                        'FPR':(sub[col].gt(0).mean() if subset=='HARD_NEGATIVE' else np.nan),
                        'positive_zero_rate':(sub[col].eq(0).mean() if subset=='POSITIVE' else np.nan)})
print(pd.DataFrame(metrics).to_string(index=False))
changed=pred.loc[pred.changed_v74].copy()
changed['delta_error']=(changed.v74_count-changed.gt_count).abs()-(changed.v73_count-changed.gt_count).abs()
print('Số ảnh thay đổi:',len(changed))
print(changed[['image_name','target','gt_count','v73_count','v74_count','delta_error']].to_string(index=False))
pred.to_csv(OUT/'v74_1301_predictions.csv',index=False)
pd.DataFrame(metrics).to_csv(OUT/'v74_1301_metrics.csv',index=False)
changed.to_csv(OUT/'v74_changed_cases.csv',index=False)
protocol={'pilot':'Exploratory, data previously examined',
          'grid':FINE_GRID,'overlap':TILE_OVERLAP_FRAC,'countgd_threshold':COUNTGD_THRESH,
          'nms_iou':NMS_IOU,'Qwen':QWEN_MODEL_ID,'min_yes':MIN_LOCAL_YES,
          'max_ratio':MAX_NEW_RATIO,
          'inference_trigger':'baseline_count=0 AND v73_count>=150 AND prior verified presence AND old tile_count>=150',
          'base_sha256':hashlib.sha256((DATA_DIR/'v73_1301_predictions.csv').read_bytes()).hexdigest()}
(OUT/'v74_protocol.json').write_text(json.dumps(protocol,indent=2),encoding='utf-8')

In [ ]:
# 15 — EXPORT; one ZIP to send back for audit
from IPython.display import FileLink,display
archive=ROOT/'V7_4_HighRes_Pilot_1301.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for file in sorted(OUT.iterdir()):
        if file.is_file() and file.suffix.lower() in ('.csv','.json','.jpg'):
            z.write(file,file.name)
print('Completed:',archive,'size MB',round(archive.stat().st_size/1e6,2))
display(FileLink(str(archive)))